# Banking Dataset Cleaning

Clean, normalize, validate, and save the messy banking transactions dataset.

In [ ]:
import numpy as np
import pandas as pd

input_path = "Banking_Messy_36.csv"
df = pd.read_csv(input_path)
print("Loaded shape:", df.shape)

Loaded shape: (36, 7)


## Standardize Text and Missing Values

In [5]:
str_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
for col in str_cols:
    df[col] = df[col].astype("string").str.strip()

missing_tokens = {"", "na", "n/a", "null", "none", "unknown", "not available", "-"}
for col in str_cols:
    df[col] = df[col].mask(df[col].str.lower().isin(missing_tokens))

print("Missing values after standardization:")
print(df.isna().sum())

Missing values after standardization:
TransactionID      0
AccountID          0
TransactionDate    0
TransactionType    0
Amount             0
PaymentMode        0
Status             0
dtype: int64


## Normalize Transaction Labels

In [ ]:
df["TransactionType"] = df["TransactionType"].str.title()

print("Transaction types:", sorted(df["TransactionType"].dropna().unique()))

Transaction types: ['Credit', 'Debit', 'Deposit', 'Refund', 'Reversal', 'Withdrawal']
Payment modes: ['ATM', 'Credit Card', 'IMPS', 'NEFT', 'UPI']
Statuses: ['Failed', 'Pending', 'Success']


In [ ]:
payment_map = {
    "atm": "ATM",
    "imps": "IMPS",
    "neft": "NEFT",
    "upi": "UPI",
    "credit card": "Credit Card",
}
df["PaymentMode"] = df["PaymentMode"].str.lower().map(payment_map)

print("Payment modes:", sorted(df["PaymentMode"].dropna().unique()))

In [ ]:
df["Status"] = df["Status"].str.title()

print("Statuses:", sorted(df["Status"].dropna().unique()))

## Normalize Amount and Banking Sign Conventions

In [7]:
df["Amount"] = (
    df["Amount"]
    .astype("string")
    .str.replace(r"(?i)rs\.?\s*", "", regex=True)
    .str.replace(",", "", regex=False)
    .str.replace(r"[^0-9.\-]", "", regex=True)
)
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
invalid_amount_count = int(df["Amount"].isna().sum())

amount_median_by_type = df.groupby("TransactionType")["Amount"].transform("median")
df["Amount"] = df["Amount"].fillna(amount_median_by_type).abs()
df.loc[df["TransactionType"].isin({"Debit", "Withdrawal"}), "Amount"] *= -1

print("Invalid amounts imputed:", invalid_amount_count)
print(df["Amount"].describe())

Invalid amounts imputed: 1
count           36.0
mean     2406.944444
std      7097.405498
min         -15500.0
25%           -587.5
50%           1025.0
75%           6800.0
max          15500.0
Name: Amount, dtype: Float64


## Normalize Transaction Dates

In [8]:
raw_dates = df["TransactionDate"].astype("string")
iso_mask = raw_dates.str.match(r"^\d{4}-\d{2}-\d{2}$")
parsed_dates = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
parsed_dates.loc[iso_mask] = pd.to_datetime(
    raw_dates.loc[iso_mask], format="%Y-%m-%d", errors="coerce"
)
parsed_dates.loc[~iso_mask] = pd.to_datetime(
    raw_dates.loc[~iso_mask], format="mixed", dayfirst=True, errors="coerce"
)
df["TransactionDate"] = parsed_dates.dt.strftime("%Y-%m-%d")

print("Invalid or missing dates:", df["TransactionDate"].isna().sum())

Invalid or missing dates: 0


## Remove Duplicate Transactions and Validate

In [9]:
duplicate_count = int(df["TransactionID"].duplicated().sum())
df = df.drop_duplicates(subset="TransactionID", keep="first").reset_index(drop=True)

assert df["TransactionID"].is_unique
assert df["TransactionDate"].notna().all()
assert df["Amount"].notna().all()
assert df.loc[df["TransactionType"].isin({"Debit", "Withdrawal"}), "Amount"].lt(0).all()
assert df.loc[df["TransactionType"].isin({"Credit", "Deposit", "Refund", "Reversal"}), "Amount"].ge(0).all()
assert df["TransactionType"].isin(["Credit", "Debit", "Deposit", "Refund", "Reversal", "Withdrawal"]).all()
assert df["PaymentMode"].isin(["ATM", "IMPS", "NEFT", "UPI", "Credit Card"]).all()
assert df["Status"].isin(["Success", "Pending", "Failed"]).all()

print("Duplicate transactions removed:", duplicate_count)
print("Validated shape:", df.shape)
print("Missing values:")
print(df.isna().sum())

Duplicate transactions removed: 0
Validated shape: (36, 7)
Missing values:
TransactionID      0
AccountID          0
TransactionDate    0
TransactionType    0
Amount             0
PaymentMode        0
Status             0
dtype: int64


## Save the Cleaned Dataset

In [ ]:
output_path = "Banking_Cleaned_36.csv"
df.to_csv(output_path, index=False)

cleaned_df = pd.read_csv(output_path)
print("Saved to:", output_path)
print("Saved shape:", cleaned_df.shape)
print(cleaned_df.head())

Saved to: c:\Users\Dharshan\Desktop\Data_Cleaning\Banking\Banking_Cleaned_36.csv
Saved shape: (36, 7)
  TransactionID AccountID TransactionDate TransactionType   Amount  \
0       TXN9001   ACC7001      2024-05-16          Credit   9999.0   
1       TXN9002   ACC7003      2024-05-31          Credit  15500.0   
2       TXN9003   ACC7003      2024-05-24          Credit    500.0   
3       TXN9004   ACC7009      2024-05-24      Withdrawal -15500.0   
4       TXN9005   ACC7010      2024-05-03          Credit    500.0   

  PaymentMode   Status  
0         ATM  Success  
1         ATM  Success  
2         ATM  Success  
3        IMPS  Success  
4        IMPS  Success  
